# Proyecto 1: Análisis de Datos con LisPy + Python

**Objetivo**: Pipeline completo de análisis de datos usando LisPy como orquestador y Python (pandas, numpy, matplotlib) como motor de cálculo.

**Features demostradas**:
- `py:run` para código Python multi-línea
- `py:import` con caché persistente entre celdas
- `py:->lisp` para traer resultados a Lisp
- `py:getitem` para indexar objetos Python
- Símbolos auto-resueltos (`pandas`, `numpy`, `math`)
- Persistencia de sesión con `py:save-session` / `py:load-session`
- Visualización con matplotlib desde Lisp

**Dataset**: Generamos datos sintéticos de ventas (1000 registros) para evitar dependencias externas.

## 1. Inicialización y Carga de Librerías

In [ ]:
;; Cargar librerías necesarias
(load "lib/persist.lp")
(py:import "numpy")
(py:import "pandas")
(py:import "matplotlib.pyplot")
(py:import "datetime")
(defvar plt (py:import "matplotlib.pyplot"))

## 2. Generación de Datos Sintéticos

Creamos un dataset de ventas con:
- 4 categorías de productos
- 3 regiones
- 12 meses de datos
- Precios y cantidades con distribución normal

In [ ]:
(py:run "
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

np.random.seed(42)
n = 1000

# Generar datos sintéticos
categorias = np.random.choice(['Electronica', 'Ropa', 'Alimentos', 'Libros'], n)
regiones = np.random.choice(['Norte', 'Sur', 'Este'], n)
fechas = [datetime(2025, 1, 1) + timedelta(days=int(d)) for d in np.random.randint(0, 365, n)]
precios = np.round(np.random.normal(100, 30, n), 2)
cantidades = np.random.randint(1, 20, n)
descuentos = np.round(np.random.uniform(0, 0.3, n), 2)

df = pd.DataFrame({
    'fecha': fechas,
    'categoria': categorias,
    'region': regiones,
    'precio_unitario': precios,
    'cantidad': cantidades,
    'descuento': descuentos
})

# Calcular columnas derivadas
df['total'] = df['precio_unitario'] * df['cantidad']
df['total_con_descuento'] = df['total'] * (1 - df['descuento'])
df['mes'] = df['fecha'].dt.month
df['trimestre'] = df['fecha'].dt.quarter

print(f'Dataset creado: {len(df)} registros')
print(df.head(10).to_string())
")

## 3. Estadísticas Descriptivas desde Lisp

Accedemos al DataFrame desde Lisp usando `@df` y extraemos estadísticas.

In [ ]:
;; Acceder al DataFrame desde Lisp
(defvar *df* @df)

;; Obtener estadísticas descriptivas
(py:run "print(df.describe().to_string())")

;; Obtener info de columnas
(py:run "print('\\nColumnas:', list(df.columns))")
(py:run "print('Tipos:\\n', df.dtypes.to_string())")

## 4. Análisis por Categoría — GROUP BY desde Lisp

Calculamos ventas totales, promedio, y conteo por categoría.

In [ ]:
(py:run "
print('=== Ventas por Categoría ===')
resumen_cat = df.groupby('categoria').agg(
    total_ventas=('total_con_descuento', 'sum'),
    promedio_venta=('total_con_descuento', 'mean'),
    cantidad_operaciones=('categoria', 'count'),
    precio_promedio=('precio_unitario', 'mean'),
    descuento_promedio=('descuento', 'mean')
).round(2)
print(resumen_cat.to_string())
")

## 5. Análisis por Región y Trimestre

Ventas cruzadas: cómo se distribuyen las ventas por región y trimestre.

In [ ]:
(py:run "
print('=== Ventas por Región y Trimestre ===')
pivot = df.pivot_table(
    values='total_con_descuento',
    index='region',
    columns='trimestre',
    aggfunc='sum'
).round(2)
print(pivot.to_string())

print('\\n=== Top 5 Meses por Ventas Totales ===')
ventas_mes = df.groupby('mes')['total_con_descuento'].sum().sort_values(ascending=False)
print(ventas_mes.head(5).to_string())
")

## 6. Filtrado y Consultas desde Lisp

Usamos Lisp para generar filtros dinámicos que se aplican en pandas.

In [ ]:
;; Definir criterio de filtro en Lisp
(defun ventas-altas-p (limite)
  (format nil "df[df['total_con_descuento'] > ~A]" limite))

;; Aplicar filtro desde Lisp
(py:run (ventas-altas-p 500))
(py:run (format nil "print('Transacciones > $500: ' + str(len(~A)))" (ventas-altas-p 500)))

;; Filtrar por categoría específica desde Lisp
(defun filtrar-categoria (categoria)
  (format nil "df[df['categoria'] == '~A']" categoria))

(py:run (format nil "
electronica = ~A
print(f'Electrónica: {len(electronica)} ventas, Total: ${electronica[\"total_con_descuento\"].sum():.2f}')" (filtrar-categoria "Electronica")))

## 7. Visualización con Matplotlib

Generamos gráficos desde Lisp usando matplotlib.

In [ ]:
;; Gráfico 1: Ventas por categoría (barras)
(py:run "
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# 1. Barras: ventas por categoría
resumen_cat['total_ventas'].plot(kind='bar', ax=axes[0,0], color='steelblue')
axes[0,0].set_title('Ventas Totales por Categoría')
axes[0,0].set_ylabel('Total ($)')
axes[0,0].tick_params(axis='x', rotation=45)

# 2. Pie: distribución de categorías
resumen_cat['cantidad_operaciones'].plot(kind='pie', ax=axes[0,1], autopct='%1.1f%%')
axes[0,1].set_title('Distribución de Operaciones')
axes[0,1].set_ylabel('')

# 3. Líneas: ventas mensuales
for region in df['region'].unique():
    data = df[df['region'] == region].groupby('mes')['total_con_descuento'].sum()
    axes[1,0].plot(data.index, data.values, marker='o', label=region)
axes[1,0].set_title('Ventas Mensuales por Región')
axes[1,0].set_xlabel('Mes')
axes[1,0].set_ylabel('Total ($)')
axes[1,0].legend()
axes[1,0].set_xticks(range(1, 13))

# 4. Scatter: precio vs cantidad
axes[1,1].scatter(df['precio_unitario'], df['cantidad'], alpha=0.5, c=df['trimestre'], cmap='viridis')
axes[1,1].set_title('Precio vs Cantidad (color=trimestre)')
axes[1,1].set_xlabel('Precio Unitario ($)')
axes[1,1].set_ylabel('Cantidad')

plt.tight_layout()
plt.savefig('analisis-ventas.png', dpi=100)
plt.show()
print('Gráfico guardado: analisis-ventas.png')
")

## 8. Resultados Agregados a Lisp

Traemos los resultados calculados en Python al mundo Lisp para procesamiento posterior.

In [ ]:
;; Extraer estadísticas a Lisp para análisis
(py:run "
# Preparamos datos para consumir desde Lisp
totales_por_categoria = resumen_cat['total_ventas'].to_dict()
venta_total = df['total_con_descuento'].sum()
venta_promedio = df['total_con_descuento'].mean()
categoria_top = resumen_cat['total_ventas'].idxmax()
")

;; Leer resultados desde Lisp
(let ((total @venta_total)
      (promedio @venta_promedio)
      (top @categoria_top))
  (format t "=== Resumen General ===~%")
  (format t "Venta Total: $~,2F~%" total)
  (format t "Venta Promedio: $~,2F~%" promedio)
  (format t "Categoría Top: ~A~%" top))

## 9. Persistencia de la Sesión

Guardamos el DataFrame y los resultados para continuar en otra sesión.

In [ ]:
;; Guardar sesión para checkpoint
(py:save-session "checkpoint-analisis.txt")
(format t "Sesión guardada en checkpoint-analisis.txt~%")
(format t "Para restaurar: (py:load-session \"checkpoint-analisis.txt\")~%")

## 10. Ejercicios Propuestos

1. Agregar una columna `margen` = `total_con_descuento * 0.25` y calcular margen por región
2. Identificar los 10 productos más vendidos (simular IDs de producto)
3. Crear un reporte automático en formato texto con `format` de Lisp
4. Usar `py:save-session` para hacer checkpoint entre pasos del análisis
5. Convertir los resultados a hash-tables de Lisp con `py:->lisp` para procesamiento funcional